In [9]:
from pathlib import Path
import zipfile
import shutil
import os

input_root = Path("/kaggle/input")
restart_root = Path("/kaggle/working/pcgnn_clean_restart")
project = restart_root / "PC-GNN-COMP8851-Instrumented"

if restart_root.exists():
    shutil.rmtree(restart_root)

restart_root.mkdir(parents=True)

code_zips = [
    path for path in input_root.rglob("*.zip")
    if "instrumented" in path.name.lower()
]

if code_zips:
    print("Code ZIP found:", code_zips[0])

    with zipfile.ZipFile(code_zips[0]) as archive:
        archive.extractall(restart_root)

    extracted_projects = [
        path.parent
        for path in restart_root.rglob("main.py")
        if (
            path.parent /
            "config/pcgnn_yelpchi_smoke.yml"
        ).exists()
    ]

    assert extracted_projects, (
        "The code ZIP was extracted, but the PC-GNN project was not found."
    )

    project = extracted_projects[0]

else:
    print("No code ZIP found. Checking automatically extracted files.")

    extracted_projects = [
        path.parent
        for path in input_root.rglob("main.py")
        if (
            path.parent /
            "config/pcgnn_yelpchi_smoke.yml"
        ).exists()
    ]

    if not extracted_projects:
        print("\nAvailable input files:")
        for path in list(input_root.rglob("*"))[:100]:
            if path.is_file():
                print(path)

        raise FileNotFoundError(
            "The attached PC-GNN project could not be located."
        )

    source_project = extracted_projects[0]

    print("Extracted project found:", source_project)

    shutil.copytree(
        source_project,
        project,
        dirs_exist_ok=True,
    )

data_directory = project / "data"
data_directory.mkdir(parents=True, exist_ok=True)

required_files = [
    "YelpChi.mat",
    "yelp_homo_adjlists.pickle",
    "yelp_rur_adjlists.pickle",
    "yelp_rtr_adjlists.pickle",
    "yelp_rsr_adjlists.pickle",
]

data_zips = [
    path for path in input_root.rglob("*.zip")
    if "yelpchi-data" in path.name.lower()
]

if data_zips:
    print("Data ZIP found:", data_zips[0])

    with zipfile.ZipFile(data_zips[0]) as archive:
        archive.extractall(data_directory)

for filename in required_files:
    destination = data_directory / filename

    if not destination.exists():
        matches = list(input_root.rglob(filename))

        if matches:
            shutil.copy2(matches[0], destination)

os.chdir(project)

print("\nProject:", project)
print("main.py:", Path("main.py").exists())

all_present = True

for filename in required_files:
    path = data_directory / filename
    exists = path.exists()
    all_present = all_present and exists
    size_mb = path.stat().st_size / (1024 ** 2) if exists else 0

    print(f"{filename}: {exists} ({size_mb:.2f} MB)")

assert Path("main.py").exists(), "main.py is missing."
assert all_present, "One or more YelpChi files are missing."

print("\nPROJECT AND DATA READY: True")

No code ZIP found. Checking automatically extracted files.
Extracted project found: /kaggle/input/datasets/maruf009/pcgnn-comp8851/PC-GNN-COMP8851-Instrumented

Project: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented
main.py: True
YelpChi.mat: True (198.06 MB)
yelp_homo_adjlists.pickle: True (111.57 MB)
yelp_rur_adjlists.pickle: True (2.90 MB)
yelp_rtr_adjlists.pickle: True (17.90 MB)
yelp_rsr_adjlists.pickle: True (98.86 MB)

PROJECT AND DATA READY: True


In [10]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "GPU is not enabled."

gpu_name = torch.cuda.get_device_name(0)
gpu_memory = (
    torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
)

print("GPU:", gpu_name)
print(f"GPU memory: {gpu_memory:.2f} GB")
print("Visible GPU count:", torch.cuda.device_count())

assert "T4" in gpu_name.upper(), (
    f"Expected T4 but received {gpu_name}"
)

print("\nGPU READY: True")

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
Visible GPU count: 1

GPU READY: True


In [11]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import os

project = next(
    Path("/kaggle/working/pcgnn_clean_restart").rglob("main.py")
).parent

os.chdir(project)

required_packages = {
    "yaml": "PyYAML",
    "numpy": "numpy",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "networkx": "networkx",
}

missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *missing_packages,
    ])

import yaml
import numpy
import scipy
import sklearn
import networkx
import torch

environment_file = Path("evidence/kaggle_t4_environment.txt")
environment_file.parent.mkdir(parents=True, exist_ok=True)

freeze_output = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True,
)

environment_file.write_text(
    "\n".join([
        f"Python: {sys.version}",
        f"PyTorch: {torch.__version__}",
        f"CUDA: {torch.version.cuda}",
        f"GPU: {torch.cuda.get_device_name(0)}",
        f"NumPy: {numpy.__version__}",
        f"SciPy: {scipy.__version__}",
        f"scikit-learn: {sklearn.__version__}",
        f"NetworkX: {networkx.__version__}",
        "",
        "INSTALLED PACKAGES",
        freeze_output,
    ]),
    encoding="utf-8",
)

print("Working directory:", Path.cwd())
print("Environment saved:", environment_file)
print("\nENVIRONMENT READY: True")

Working directory: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented
Environment saved: evidence/kaggle_t4_environment.txt

ENVIRONMENT READY: True


In [12]:
from pathlib import Path
import yaml

project = next(
    Path("/kaggle/working/pcgnn_clean_restart").rglob("main.py")
).parent

source_config = project / "config/pcgnn_yelpchi_smoke.yml"
smoke_config = (
    project / "config/pcgnn_yelpchi_smoke_kaggle_t4.yml"
)

assert source_config.exists()

with source_config.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config["no_cuda"] = False
config["cuda_id"] = "0"
config["num_epochs"] = 2
config["valid_epochs"] = 1
config["seed"] = 72
config["run_mode"] = "smoke_test_kaggle_t4"
config["results_dir"] = "./results/smoke-test-kaggle-t4"

with smoke_config.open("w", encoding="utf-8") as file:
    yaml.safe_dump(config, file, sort_keys=False)

print(smoke_config.read_text())

print("\nGPU SMOKE CONFIG READY: True")

data_name: yelp
data_dir: ./data/
train_ratio: 0.4
test_ratio: 0.67
save_dir: ./pytorch_models/
split_path: null
run_mode: smoke_test_kaggle_t4
results_dir: ./results/smoke-test-kaggle-t4
model: PCGNN
multi_relation: GNN
emb_size: 64
thres: 0.5
rho: 0.5
seed: 72
optimizer: adam
lr: 0.01
weight_decay: 0.001
batch_size: 1024
num_epochs: 2
valid_epochs: 1
alpha: 2
no_cuda: false
cuda_id: '0'


GPU SMOKE CONFIG READY: True


In [13]:
from pathlib import Path
import subprocess
import sys
import os

project = next(
    Path("/kaggle/working/pcgnn_clean_restart").rglob("main.py")
).parent

os.chdir(project)

config_path = Path(
    "config/pcgnn_yelpchi_smoke_kaggle_t4.yml"
)
log_path = Path("pcgnn_smoke_kaggle_t4.log")

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"

command = [
    sys.executable,
    "main.py",
    "--config",
    str(config_path),
]

with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    )

    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
        log_file.flush()

    return_code = process.wait()

print("\nReturn code:", return_code)

assert return_code == 0, (
    "Smoke test failed. Stop and send the output."
)

print("\nGPU SMOKE TEST COMPLETED: True")

/kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/src/model_handler.py:258: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  batch_loss = gnn_model.loss(batch_nodes, Variable(torch.cuda.LongTensor(batch_label)))
**************** MODEL CONFIGURATION ****************
alpha                    -->   2
batch_size               -->   1024
cuda_id                  -->   0
data_dir                 -->   ./data/
data_name                -->   yelp
emb_size                 -->   64
lr                       -->   0.01
model                    -->   PCGNN
multi_relation           -->   GNN
no_cuda                  -->   False
num_epochs               -->   2
optimizer                -->   adam
results_dir              -->   ./results/smoke-test-kaggle-t4
rho                      -->

In [14]:
from pathlib import Path
import json
import csv

project = next(
    Path("/kaggle/working/pcgnn_clean_restart").rglob("main.py")
).parent

results_root = project / "results/smoke-test-kaggle-t4"

summary_files = list(results_root.rglob("summary.json"))
assert summary_files, "No summary.json was created."

summary_path = max(
    summary_files,
    key=lambda path: path.stat().st_mtime,
)

run_directory = summary_path.parent
epoch_file = run_directory / "epoch_times.csv"

with summary_path.open("r", encoding="utf-8") as file:
    summary = json.load(file)

with epoch_file.open("r", encoding="utf-8") as file:
    epoch_rows = list(csv.DictReader(file))

print("Run ID:", summary["run_id"])
print("CUDA used:", summary["environment"]["cuda_used"])
print("GPU:", summary["environment"]["gpu_name"])
print("Epochs:", summary["timing"]["training_epochs"])
print("Mean epoch time:", summary["timing"]["mean_train_epoch_seconds"])
print("Total epoch time:", summary["timing"]["total_train_epoch_seconds"])
print("Peak GPU memory:", summary["timing"]["peak_gpu_memory_mb"])
print("Inference time:", summary["timing"]["test_inference_seconds"])
print("AUROC:", summary["test_metrics"]["auroc"])
print("AUPRC:", summary["test_metrics"]["auprc"])
print("Macro-F1:", summary["test_metrics"]["macro_f1"])
print("Fraud F1:", summary["test_metrics"]["fraud_f1"])
print("G-Mean:", summary["test_metrics"]["gmean"])

assert summary["environment"]["cuda_used"] is True
assert "T4" in summary["environment"]["gpu_name"].upper()
assert summary["timing"]["training_epochs"] == 2
assert len(epoch_rows) == 2

print("\nGPU SMOKE RESULTS VERIFIED: True")

Run ID: 20260908T121725359413Z_yelp_PCGNN_smoke_test_kaggle_t4_seed72
CUDA used: True
GPU: Tesla T4
Epochs: 2
Mean epoch time: 12.441009108499657
Total epoch time: 24.882018216999313
Peak GPU memory: 286.39794921875
Inference time: 23.319375842001136
AUROC: 0.7504866287119788
AUPRC: 0.3473193580259838
Macro-F1: 0.5582326212393416
Fraud F1: 0.3764943902887622
G-Mean: 0.6826101470983609

GPU SMOKE RESULTS VERIFIED: True


In [15]:
from pathlib import Path
import zipfile

project = next(
    Path("/kaggle/working/pcgnn_clean_restart").rglob("main.py")
).parent

output_zip = Path(
    "/kaggle/working/pcgnn_kaggle_t4_smoke_results_clean.zip"
)

items = [
    project / "results/smoke-test-kaggle-t4",
    project / "pcgnn_smoke_kaggle_t4.log",
    project / "config/pcgnn_yelpchi_smoke_kaggle_t4.yml",
    project / "evidence/kaggle_t4_environment.txt",
]

with zipfile.ZipFile(
    output_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for item in items:
        assert item.exists(), f"Missing: {item}"

        if item.is_dir():
            for file in item.rglob("*"):
                if file.is_file():
                    archive.write(
                        file,
                        file.relative_to(project),
                    )
        else:
            archive.write(
                item,
                item.relative_to(project),
            )

print("Created:", output_zip)
print(f"Size: {output_zip.stat().st_size / (1024 ** 2):.2f} MB")
print("\nSMOKE PACKAGE READY: True")

Created: /kaggle/working/pcgnn_kaggle_t4_smoke_results_clean.zip
Size: 4.67 MB

SMOKE PACKAGE READY: True


In [16]:
from pathlib import Path
import yaml

project = next(
    Path("/kaggle/working/pcgnn_clean_restart").rglob("main.py")
).parent

source_config = (
    project / "config/pcgnn_yelpchi_author_reproduction.yml"
)

full_config = (
    project /
    "config/pcgnn_yelpchi_author_kaggle_t4_100epochs.yml"
)

assert source_config.exists(), (
    "Author-reproduction configuration was not found."
)

with source_config.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config["no_cuda"] = False
config["cuda_id"] = "0"
config["num_epochs"] = 100
config["valid_epochs"] = 5
config["seed"] = 72
config["split_path"] = None
config["train_ratio"] = 0.4
config["thres"] = 0.5
config["run_mode"] = "author_reproduction_kaggle_t4_32d_100e"
config["results_dir"] = "./results/author-reproduction-kaggle-t4"

with full_config.open("w", encoding="utf-8") as file:
    yaml.safe_dump(config, file, sort_keys=False)

print(full_config.read_text())

assert config["no_cuda"] is False
assert config["num_epochs"] == 100
assert config["valid_epochs"] == 5
assert config["seed"] == 72

print("\nFULL AUTHOR CONFIG READY: True")

data_name: yelp
data_dir: ./data/
train_ratio: 0.4
test_ratio: 0.67
save_dir: ./pytorch_models/
split_path: null
run_mode: author_reproduction_kaggle_t4_32d_100e
results_dir: ./results/author-reproduction-kaggle-t4
model: PCGNN
multi_relation: GNN
emb_size: 64
thres: 0.5
rho: 0.5
seed: 72
optimizer: adam
lr: 0.01
weight_decay: 0.001
batch_size: 1024
num_epochs: 100
valid_epochs: 5
alpha: 2
no_cuda: false
cuda_id: '0'


FULL AUTHOR CONFIG READY: True


In [17]:
from pathlib import Path
import subprocess
import sys
import os
import time

project = next(
    Path("/kaggle/working/pcgnn_clean_restart").rglob("main.py")
).parent

os.chdir(project)

config_path = Path(
    "config/pcgnn_yelpchi_author_kaggle_t4_100epochs.yml"
)

log_path = Path(
    "pcgnn_author_kaggle_t4_100epochs.log"
)

assert config_path.exists(), (
    "The 100-epoch configuration was not found."
)

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"

command = [
    sys.executable,
    "main.py",
    "--config",
    str(config_path),
]

print("Starting full PC-GNN run")
print("Configuration:", config_path)
print("GPU: Tesla T4")
print("Epochs: 100")
print("Seed: 72")
print("This may take approximately 25–35 minutes.\n")

started = time.perf_counter()

with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    )

    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
        log_file.flush()

    return_code = process.wait()

elapsed = time.perf_counter() - started

print("\nReturn code:", return_code)
print(f"Total command time: {elapsed:.2f} seconds")
print("Log saved:", log_path)

assert return_code == 0, (
    "The 100-epoch run failed. Stop and send me the complete output."
)

print("\nFULL 100-EPOCH RUN COMPLETED: True")

Starting full PC-GNN run
Configuration: config/pcgnn_yelpchi_author_kaggle_t4_100epochs.yml
GPU: Tesla T4
Epochs: 100
Seed: 72
This may take approximately 25–35 minutes.

/kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/src/model_handler.py:258: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  batch_loss = gnn_model.loss(batch_nodes, Variable(torch.cuda.LongTensor(batch_label)))
**************** MODEL CONFIGURATION ****************
alpha                    -->   2
batch_size               -->   1024
cuda_id                  -->   0
data_dir                 -->   ./data/
data_name                -->   yelp
emb_size                 -->   64
lr                       -->   0.01
model                    -->   PCGNN
multi_relation           -->   GNN
no_cuda                  

In [18]:
from pathlib import Path
import json
import csv

print("[1/5] Locating the completed author-reproduction run...")

project = next(
    Path("/kaggle/working/pcgnn_clean_restart").rglob("main.py")
).parent

results_root = (
    project / "results/author-reproduction-kaggle-t4"
)

summary_files = list(results_root.rglob("summary.json"))

assert summary_files, (
    "No summary.json was found for the 100-epoch run."
)

summary_path = max(
    summary_files,
    key=lambda path: path.stat().st_mtime,
)

run_directory = summary_path.parent
epoch_file = run_directory / "epoch_times.csv"
config_file = run_directory / "run_config.json"
metrics_file = run_directory / "test_metrics.csv"

print("[2/5] Loading configuration, metrics and epoch timing...")

assert epoch_file.exists(), "epoch_times.csv is missing."
assert config_file.exists(), "run_config.json is missing."
assert metrics_file.exists(), "test_metrics.csv is missing."

with summary_path.open("r", encoding="utf-8") as file:
    summary = json.load(file)

with epoch_file.open("r", encoding="utf-8") as file:
    epoch_rows = list(csv.DictReader(file))

print("[3/5] Checking GPU and completed epochs...")

cuda_used = summary["environment"]["cuda_used"]
gpu_name = summary["environment"]["gpu_name"]
completed_epochs = summary["timing"]["training_epochs"]

print("CUDA used:", cuda_used)
print("GPU:", gpu_name)
print("Completed epochs:", completed_epochs)
print("Saved epoch rows:", len(epoch_rows))

assert cuda_used is True, "The full run did not use CUDA."
assert gpu_name and "T4" in gpu_name.upper(), (
    "The full run did not use a Tesla T4."
)
assert completed_epochs == 100, (
    f"Expected 100 epochs but found {completed_epochs}."
)
assert len(epoch_rows) == 100, (
    f"Expected 100 timing rows but found {len(epoch_rows)}."
)

print("[4/5] Printing saved training progress...")

for row in epoch_rows:
    epoch_number = int(row["epoch"]) + 1

    if epoch_number == 1 or epoch_number % 10 == 0:
        train_seconds = float(row["train_seconds"])
        loss = float(row["mean_batch_loss"])

        message = (
            f"Epoch {epoch_number:3d}/100 | "
            f"Loss: {loss:.6f} | "
            f"Train time: {train_seconds:.3f}s"
        )

        validation_auroc = row.get("validation_auroc", "")

        if validation_auroc not in ("", None):
            message += (
                f" | Validation AUROC: "
                f"{float(validation_auroc):.4f}"
            )

        print(message)

print("[5/5] Final run summary")

timing = summary["timing"]
metrics = summary["test_metrics"]
configuration = summary["configuration"]
split = summary["split"]

print("\nRUN INFORMATION")
print("Run ID:", summary["run_id"])
print("Run mode:", summary["run_mode"])
print("Dataset:", summary["dataset"])
print("Feature dimension:", configuration.get("feature_dimension", 32))
print("Seed:", summary["seed"])
print("Best validation epoch:", summary["best_validation_epoch"])
print("Best validation AUROC:", summary["best_validation_auroc"])
print("Best validation Macro-F1:", summary["best_validation_macro_f1"])

print("\nSPLIT")
print("Training nodes:", split["train"]["nodes"])
print("Validation nodes:", split["validation"]["nodes"])
print("Test nodes:", split["test"]["nodes"])
print("Training fraud nodes:", split["train"]["fraud_nodes"])
print("Test fraud nodes:", split["test"]["fraud_nodes"])

print("\nTIMING")
print("Total training epoch time:", timing["total_train_epoch_seconds"])
print("Mean epoch time:", timing["mean_train_epoch_seconds"])
print("Median epoch time:", timing["median_train_epoch_seconds"])
print("Epoch-time standard deviation:", timing["std_train_epoch_seconds"])
print("Inference time:", timing["test_inference_seconds"])
print("Peak GPU memory MB:", timing["peak_gpu_memory_mb"])
print("Complete run time:", timing["total_run_seconds"])

print("\nTEST METRICS")
print("AUROC:", metrics["auroc"])
print("AUPRC:", metrics["auprc"])
print("Macro-F1:", metrics["macro_f1"])
print("Fraud F1:", metrics["fraud_f1"])
print("Fraud precision:", metrics["fraud_precision"])
print("Fraud recall:", metrics["fraud_recall"])
print("G-Mean:", metrics["gmean"])
print("Accuracy:", metrics["accuracy"])

print("\nFULL AUTHOR RUN VERIFIED: True")

[1/5] Locating the completed author-reproduction run...
[2/5] Loading configuration, metrics and epoch timing...
[3/5] Checking GPU and completed epochs...
CUDA used: True
GPU: Tesla T4
Completed epochs: 100
Saved epoch rows: 100
[4/5] Printing saved training progress...
Epoch   1/100 | Loss: 2.063040 | Train time: 12.772s | Validation AUROC: 0.7312
Epoch  10/100 | Loss: 1.904271 | Train time: 11.758s
Epoch  20/100 | Loss: 1.881777 | Train time: 12.123s
Epoch  30/100 | Loss: 1.826077 | Train time: 12.298s
Epoch  40/100 | Loss: 1.805970 | Train time: 12.202s
Epoch  50/100 | Loss: 1.790844 | Train time: 11.569s
Epoch  60/100 | Loss: 1.768645 | Train time: 12.334s
Epoch  70/100 | Loss: 1.748076 | Train time: 12.173s
Epoch  80/100 | Loss: 1.721138 | Train time: 12.297s
Epoch  90/100 | Loss: 1.741606 | Train time: 11.719s
Epoch 100/100 | Loss: 1.704330 | Train time: 11.732s
[5/5] Final run summary

RUN INFORMATION
Run ID: 20260908T122216039896Z_yelp_PCGNN_author_reproduction_kaggle_t4_32d_1

In [19]:
from pathlib import Path
import zipfile

print("[1/4] Locating PC-GNN project and completed results...")

project = next(
    Path("/kaggle/working/pcgnn_clean_restart").rglob("main.py")
).parent

output_zip = Path(
    "/kaggle/working/"
    "pcgnn_kaggle_t4_author_100epochs_results.zip"
)

items_to_include = [
    project / "results/author-reproduction-kaggle-t4",
    project / "pcgnn_author_kaggle_t4_100epochs.log",
    project / "config/pcgnn_yelpchi_author_kaggle_t4_100epochs.yml",
    project / "evidence/kaggle_t4_environment.txt",
    project / "pcgnn_commit.txt",
    project / "docs/PAPER_AND_REPRODUCTION_AUDIT.md",
]

for item in items_to_include:
    assert item.exists(), f"Missing required item: {item}"

print("[2/4] Creating the result ZIP...")

files_added = 0

with zipfile.ZipFile(
    output_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for item in items_to_include:
        if item.is_dir():
            for file in item.rglob("*"):
                if file.is_file():
                    archive.write(
                        file,
                        arcname=file.relative_to(project),
                    )
                    files_added += 1
        else:
            archive.write(
                item,
                arcname=item.relative_to(project),
            )
            files_added += 1

print("[3/4] Verifying the ZIP...")

with zipfile.ZipFile(output_zip, "r") as archive:
    corrupt_file = archive.testzip()
    archived_files = archive.namelist()

assert corrupt_file is None, (
    f"ZIP verification failed at: {corrupt_file}"
)

print("[4/4] Package complete")
print("Files included:", files_added)
print("Files verified:", len(archived_files))
print("Output:", output_zip)
print(f"ZIP size: {output_zip.stat().st_size / (1024 ** 2):.2f} MB")

print("\nFULL AUTHOR RESULTS PACKAGE READY: True")

[1/4] Locating PC-GNN project and completed results...
[2/4] Creating the result ZIP...
[3/4] Verifying the ZIP...
[4/4] Package complete
Files included: 12
Files verified: 12
Output: /kaggle/working/pcgnn_kaggle_t4_author_100epochs_results.zip
ZIP size: 4.69 MB

FULL AUTHOR RESULTS PACKAGE READY: True


In [20]:
from pathlib import Path
from scipy.io import loadmat
import numpy as np
import hashlib
import json
import csv
import shutil

print("=" * 70)
print("STEP 1: LOCATING THE PC-GNN PROJECT")
print("=" * 70)

project_candidates = [
    Path("/kaggle/working/PC-GNN-COMP8851-Instrumented"),
    Path.cwd(),
]

project_dir = next(
    (
        path for path in project_candidates
        if (path / "main.py").exists()
    ),
    None,
)

assert project_dir is not None, (
    "Could not locate PC-GNN-COMP8851-Instrumented."
)

print("Project directory:", project_dir)


print("\n" + "=" * 70)
print("STEP 2: LOCATING THE COMPLETED 100-EPOCH SPLIT")
print("=" * 70)

split_candidates = sorted(
    (
        project_dir /
        "results" /
        "author-reproduction-kaggle-t4"
    ).glob("*/split_indices.npz"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

assert split_candidates, (
    "No split_indices.npz file was found in the completed run."
)

source_split = split_candidates[0]
print("Source split:", source_split)


print("\n" + "=" * 70)
print("STEP 3: LOADING LABELS AND THE ORIGINAL SPLIT")
print("=" * 70)

dataset_path = project_dir / "data" / "YelpChi.mat"

assert dataset_path.exists(), (
    f"YelpChi.mat was not found at {dataset_path}"
)

labels = loadmat(dataset_path)["label"].flatten().astype(np.int64)
total_nodes = len(labels)

source = np.load(source_split)

base_train = np.asarray(source["train_idx"], dtype=np.int64)
valid_idx = np.asarray(source["valid_idx"], dtype=np.int64)
test_idx = np.asarray(source["test_idx"], dtype=np.int64)

print("Total labelled nodes:", total_nodes)
print("Original training nodes:", len(base_train))
print("Fixed validation nodes:", len(valid_idx))
print("Fixed test nodes:", len(test_idx))
print("Fraud nodes in original training set:", int(labels[base_train].sum()))


print("\n" + "=" * 70)
print("STEP 4: CHECKING THE SOURCE SPLIT")
print("=" * 70)

assert len(np.unique(base_train)) == len(base_train)
assert len(np.unique(valid_idx)) == len(valid_idx)
assert len(np.unique(test_idx)) == len(test_idx)

assert not np.intersect1d(base_train, valid_idx).size
assert not np.intersect1d(base_train, test_idx).size
assert not np.intersect1d(valid_idx, test_idx).size

all_source_nodes = np.concatenate(
    [base_train, valid_idx, test_idx]
)

assert len(np.unique(all_source_nodes)) == total_nodes

print("No duplicated node IDs: PASS")
print("No overlap between train/validation/test: PASS")
print("All labelled nodes are accounted for: PASS")


print("\n" + "=" * 70)
print("STEP 5: BUILDING NESTED STRATIFIED TRAINING SETS")
print("=" * 70)

split_seed = 2
rng = np.random.RandomState(split_seed)

fraud_train = base_train[labels[base_train] == 1].copy()
normal_train = base_train[labels[base_train] == 0].copy()

rng.shuffle(fraud_train)
rng.shuffle(normal_train)

base_fraud_fraction = len(fraud_train) / len(base_train)

target_sizes = {
    "TR10": round(total_nodes * 0.10),
    "TR20": round(total_nodes * 0.20),
    "TR30": round(total_nodes * 0.30),
    "TR40": len(base_train),
}

split_sets = {}

for split_name, target_total in target_sizes.items():

    if split_name == "TR40":
        selected_train = np.sort(base_train.copy())
    else:
        target_fraud = round(
            target_total * base_fraud_fraction
        )
        target_normal = target_total - target_fraud

        selected_train = np.sort(
            np.concatenate(
                [
                    fraud_train[:target_fraud],
                    normal_train[:target_normal],
                ]
            )
        )

    unused_idx = np.sort(
        np.setdiff1d(
            base_train,
            selected_train,
            assume_unique=False,
        )
    )

    split_sets[split_name] = {
        "train_idx": selected_train,
        "valid_idx": np.sort(valid_idx.copy()),
        "test_idx": np.sort(test_idx.copy()),
        "unused_idx": unused_idx,
    }

print("Nested training sets created.")


print("\n" + "=" * 70)
print("STEP 6: VERIFYING NESTING AND DATA ISOLATION")
print("=" * 70)

assert set(split_sets["TR10"]["train_idx"]).issubset(
    set(split_sets["TR20"]["train_idx"])
)

assert set(split_sets["TR20"]["train_idx"]).issubset(
    set(split_sets["TR30"]["train_idx"])
)

assert set(split_sets["TR30"]["train_idx"]).issubset(
    set(split_sets["TR40"]["train_idx"])
)

for split_name, arrays in split_sets.items():

    train = arrays["train_idx"]
    valid = arrays["valid_idx"]
    test = arrays["test_idx"]
    unused = arrays["unused_idx"]

    assert not np.intersect1d(train, valid).size
    assert not np.intersect1d(train, test).size
    assert not np.intersect1d(train, unused).size
    assert not np.intersect1d(valid, test).size
    assert not np.intersect1d(valid, unused).size
    assert not np.intersect1d(test, unused).size

    combined = np.concatenate(
        [train, valid, test, unused]
    )

    assert len(np.unique(combined)) == total_nodes

print("TR10 is contained within TR20: PASS")
print("TR20 is contained within TR30: PASS")
print("TR30 is contained within TR40: PASS")
print("Validation nodes remain fixed: PASS")
print("Test nodes remain fixed: PASS")
print("No split leakage detected: PASS")


print("\n" + "=" * 70)
print("STEP 7: SAVING THE SHARED SPLIT FILES")
print("=" * 70)

output_dir = (
    project_dir /
    "shared" /
    "splits" /
    "yelpchi"
)

output_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
manifest = {
    "dataset": "YelpChi",
    "split_seed": split_seed,
    "source_split": str(source_split.relative_to(project_dir)),
    "split_design": "Nested stratified training subsets",
    "nesting_rule": "TR10 subset TR20 subset TR30 subset TR40",
    "validation_fixed": True,
    "test_fixed": True,
    "splits": {},
}

for split_name, arrays in split_sets.items():

    output_file = (
        output_dir /
        f"yelpchi_{split_name.lower()}_split_seed2.npz"
    )

    np.savez_compressed(
        output_file,
        train_idx=arrays["train_idx"],
        valid_idx=arrays["valid_idx"],
        test_idx=arrays["test_idx"],
        unused_idx=arrays["unused_idx"],
    )

    train_labels = labels[arrays["train_idx"]]
    fraud_count = int(train_labels.sum())
    normal_count = int(len(train_labels) - fraud_count)

    hash_object = hashlib.sha256()

    for key in [
        "train_idx",
        "valid_idx",
        "test_idx",
        "unused_idx",
    ]:
        hash_object.update(arrays[key].tobytes())

    split_hash = hash_object.hexdigest()

    row = {
        "split": split_name,
        "train_nodes": len(arrays["train_idx"]),
        "train_percentage": (
            len(arrays["train_idx"]) /
            total_nodes * 100
        ),
        "train_fraud": fraud_count,
        "train_normal": normal_count,
        "train_fraud_percentage": (
            fraud_count /
            len(arrays["train_idx"]) * 100
        ),
        "validation_nodes": len(arrays["valid_idx"]),
        "test_nodes": len(arrays["test_idx"]),
        "unused_nodes": len(arrays["unused_idx"]),
        "sha256": split_hash,
        "file": output_file.name,
    }

    summary_rows.append(row)
    manifest["splits"][split_name] = row

    print(
        f"{split_name}: "
        f"train={row['train_nodes']:,}, "
        f"fraud={fraud_count:,}, "
        f"normal={normal_count:,}, "
        f"unused={row['unused_nodes']:,}"
    )

summary_csv = output_dir / "yelpchi_split_summary.csv"

with summary_csv.open("w", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=summary_rows[0].keys(),
    )
    writer.writeheader()
    writer.writerows(summary_rows)

manifest_path = output_dir / "yelpchi_split_manifest.json"

with manifest_path.open("w") as file:
    json.dump(manifest, file, indent=2)

archive_path = shutil.make_archive(
    "/kaggle/working/yelpchi_shared_splits",
    "zip",
    root_dir=output_dir,
)


print("\n" + "=" * 70)
print("SHARED YELPCHI SPLITS CREATED SUCCESSFULLY")
print("=" * 70)

print("Split folder:", output_dir)
print("Summary CSV:", summary_csv)
print("Manifest:", manifest_path)
print("Downloadable ZIP:", archive_path)

print("\nUse these exact split files for every eligible model.")
print("Do not regenerate them separately for each model.")

STEP 1: LOCATING THE PC-GNN PROJECT
Project directory: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented

STEP 2: LOCATING THE COMPLETED 100-EPOCH SPLIT
Source split: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/results/author-reproduction-kaggle-t4/20260908T122216039896Z_yelp_PCGNN_author_reproduction_kaggle_t4_32d_100e_seed72/split_indices.npz

STEP 3: LOADING LABELS AND THE ORIGINAL SPLIT
Total labelled nodes: 45954
Original training nodes: 18381
Fixed validation nodes: 9099
Fixed test nodes: 18474
Fraud nodes in original training set: 2671

STEP 4: CHECKING THE SOURCE SPLIT
No duplicated node IDs: PASS
No overlap between train/validation/test: PASS
All labelled nodes are accounted for: PASS

STEP 5: BUILDING NESTED STRATIFIED TRAINING SETS
Nested training sets created.

STEP 6: VERIFYING NESTING AND DATA ISOLATION
TR10 is contained within TR20: PASS
TR20 is contained within TR30: PASS
TR30 is contained within TR40: PASS
Validation nodes remain fix

In [24]:
from pathlib import Path
import numpy as np
import yaml
import os

print("=" * 70)
print("STEP 1: USING THE WRITABLE PC-GNN PROJECT")
print("=" * 70)

project_dir = Path(
    "/kaggle/working/pcgnn_clean_restart/"
    "PC-GNN-COMP8851-Instrumented"
)

assert (project_dir / "main.py").exists()

os.chdir(project_dir)

print("Project directory:", project_dir)
print("main.py found: PASS")


print("\n" + "=" * 70)
print("STEP 2: LOCATING CONFIGURATION AND TR40 SPLIT")
print("=" * 70)

source_config = (
    project_dir /
    "config" /
    "pcgnn_yelpchi_author_kaggle_t4_100epochs.yml"
)

split_path = (
    project_dir /
    "shared" /
    "splits" /
    "yelpchi" /
    "yelpchi_tr40_split_seed2.npz"
)

assert source_config.exists(), (
    f"Source configuration missing: {source_config}"
)

assert split_path.exists(), (
    f"TR40 split missing: {split_path}"
)

print("Source configuration:", source_config)
print("TR40 split:", split_path)


print("\n" + "=" * 70)
print("STEP 3: VERIFYING THE TR40 SPLIT")
print("=" * 70)

split = np.load(split_path)

train_idx = split["train_idx"]
valid_idx = split["valid_idx"]
test_idx = split["test_idx"]
unused_idx = split["unused_idx"]

assert len(train_idx) == 18381
assert len(valid_idx) == 9099
assert len(test_idx) == 18474
assert len(unused_idx) == 0

assert not np.intersect1d(train_idx, valid_idx).size
assert not np.intersect1d(train_idx, test_idx).size
assert not np.intersect1d(valid_idx, test_idx).size

print("Training nodes:", len(train_idx))
print("Validation nodes:", len(valid_idx))
print("Test nodes:", len(test_idx))
print("Unused nodes:", len(unused_idx))
print("Split verification: PASS")


print("\n" + "=" * 70)
print("STEP 4: CREATING THE UNIFIED TR40 CONFIGURATION")
print("=" * 70)

with source_config.open("r") as file:
    author_config = yaml.safe_load(file)

config = author_config.copy()

config["split_path"] = (
    "./shared/splits/yelpchi/"
    "yelpchi_tr40_split_seed2.npz"
)

config["run_mode"] = (
    "unified_yelpchi_tr40_kaggle_t4_seed72"
)

config["results_dir"] = (
    "./results/unified/yelpchi/pcgnn/tr40"
)

config["optimizer"] = "adam"
config["seed"] = 72
config["num_epochs"] = 100
config["valid_epochs"] = 5
config["no_cuda"] = False
config["cuda_id"] = "0"

output_config = (
    project_dir /
    "config" /
    "pcgnn_yelpchi_unified_tr40_seed72.yml"
)

with output_config.open("w") as file:
    yaml.safe_dump(
        config,
        file,
        sort_keys=False,
    )

print("Configuration saved:", output_config)


print("\n" + "=" * 70)
print("STEP 5: VERIFYING ARCHITECTURE SETTINGS")
print("=" * 70)

architecture_fields = [
    "model",
    "multi_relation",
    "emb_size",
    "rho",
    "alpha",
]

for field in architecture_fields:
    assert config[field] == author_config[field]
    print(f"{field}: {config[field]} — unchanged")

print("Architecture verification: PASS")


print("\n" + "=" * 70)
print("FINAL UNIFIED TR40 CONFIGURATION")
print("=" * 70)

print(output_config.read_text())

print("=" * 70)
print("UNIFIED TR40 CONFIGURATION READY: TRUE")
print("=" * 70)

print("\nTraining has NOT started yet.")

STEP 1: USING THE WRITABLE PC-GNN PROJECT
Project directory: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented
main.py found: PASS

STEP 2: LOCATING CONFIGURATION AND TR40 SPLIT
Source configuration: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/config/pcgnn_yelpchi_author_kaggle_t4_100epochs.yml
TR40 split: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/shared/splits/yelpchi/yelpchi_tr40_split_seed2.npz

STEP 3: VERIFYING THE TR40 SPLIT
Training nodes: 18381
Validation nodes: 9099
Test nodes: 18474
Unused nodes: 0
Split verification: PASS

STEP 4: CREATING THE UNIFIED TR40 CONFIGURATION
Configuration saved: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/config/pcgnn_yelpchi_unified_tr40_seed72.yml

STEP 5: VERIFYING ARCHITECTURE SETTINGS
model: PCGNN — unchanged
multi_relation: GNN — unchanged
emb_size: 64 — unchanged
rho: 0.5 — unchanged
alpha: 2 — unchanged
Architecture verification: PASS

FINAL UNIFIED TR40 CONFI

In [25]:
from pathlib import Path
import subprocess
import os
import sys
import json
import torch
from datetime import datetime, timezone

print("=" * 70)
print("PC-GNN UNIFIED TR40 RUN")
print("=" * 70)

project_dir = Path(
    "/kaggle/working/pcgnn_clean_restart/"
    "PC-GNN-COMP8851-Instrumented"
)

config_path = (
    project_dir /
    "config" /
    "pcgnn_yelpchi_unified_tr40_seed72.yml"
)

split_path = (
    project_dir /
    "shared" /
    "splits" /
    "yelpchi" /
    "yelpchi_tr40_split_seed2.npz"
)

assert (project_dir / "main.py").exists(), (
    "main.py was not found."
)

assert config_path.exists(), (
    "The unified TR40 configuration was not found."
)

assert split_path.exists(), (
    "The fixed TR40 split was not found."
)

print("Project:", project_dir)
print("Configuration:", config_path)
print("Split:", split_path)


print("\n" + "=" * 70)
print("GPU CHECK")
print("=" * 70)

assert torch.cuda.is_available(), (
    "GPU is not enabled. Enable a Kaggle GPU accelerator first."
)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2,
    ),
    "GB",
)
print("GPU 0 will be used exclusively.")


print("\n" + "=" * 70)
print("STARTING 100-EPOCH TR40 TRAINING")
print("=" * 70)

print(
    "Start time:",
    datetime.now(timezone.utc).isoformat(),
)

results_root = (
    project_dir /
    "results" /
    "unified" /
    "yelpchi" /
    "pcgnn" /
    "tr40"
)

existing_summaries = set(
    results_root.rglob("summary.json")
) if results_root.exists() else set()

log_path = Path(
    "/kaggle/working/"
    "pcgnn_unified_tr40_seed72.log"
)

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"
environment["PYTHONUNBUFFERED"] = "1"

command = [
    sys.executable,
    "-u",
    "main.py",
    "--config",
    str(config_path),
]

print("Command:", " ".join(command))
print("Live output will appear below.\n")

captured_lines = []

with log_path.open("w") as log_file:

    process = subprocess.Popen(
        command,
        cwd=project_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()
        captured_lines.append(line)

    return_code = process.wait()


print("\n" + "=" * 70)
print("TRAINING PROCESS FINISHED")
print("=" * 70)

print("Return code:", return_code)
print(
    "Finish time:",
    datetime.now(timezone.utc).isoformat(),
)
print("Saved terminal log:", log_path)

if return_code != 0:
    print("\nThe run failed. Last 30 output lines:\n")
    print("".join(captured_lines[-30:]))
    raise RuntimeError(
        "PC-GNN TR40 training did not finish successfully."
    )


print("\n" + "=" * 70)
print("LOCATING THE NEW RESULT")
print("=" * 70)

all_summaries = set(
    results_root.rglob("summary.json")
)

new_summaries = list(
    all_summaries - existing_summaries
)

if new_summaries:
    summary_path = max(
        new_summaries,
        key=lambda path: path.stat().st_mtime,
    )
else:
    summary_candidates = list(
        results_root.rglob("summary.json")
    )

    assert summary_candidates, (
        "Training succeeded, but summary.json was not found."
    )

    summary_path = max(
        summary_candidates,
        key=lambda path: path.stat().st_mtime,
    )

with summary_path.open("r") as file:
    summary = json.load(file)

metrics = summary["test_metrics"]
timing = summary["timing"]


print("\n" + "=" * 70)
print("UNIFIED TR40 RESULT")
print("=" * 70)

print("Run ID:", summary["run_id"])
print("Run mode:", summary["run_mode"])
print("Seed:", summary["seed"])
print("Best validation epoch:", summary["best_validation_epoch"])

print("\nTest metrics:")
print(f"AUROC:          {metrics['auroc']:.6f}")
print(f"AUPRC:          {metrics['auprc']:.6f}")
print(f"Macro-F1:       {metrics['macro_f1']:.6f}")
print(f"Fraud F1:       {metrics['fraud_f1']:.6f}")
print(f"Fraud precision:{metrics['fraud_precision']:.6f}")
print(f"Fraud recall:   {metrics['fraud_recall']:.6f}")
print(f"G-Mean:         {metrics['gmean']:.6f}")

print("\nTiming:")
print(
    f"Mean epoch time: "
    f"{timing['mean_train_epoch_seconds']:.3f} seconds"
)
print(
    f"Training wall time: "
    f"{timing['fit_wall_seconds_including_validation']:.3f} seconds"
)
print(
    f"Inference time: "
    f"{timing['test_inference_seconds']:.3f} seconds"
)
print(
    f"Peak GPU memory: "
    f"{timing['peak_gpu_memory_mb']:.3f} MB"
)

print("\nSummary file:", summary_path)
print("Result directory:", summary_path.parent)

print("\n" + "=" * 70)
print("UNIFIED TR40 RUN COMPLETED SUCCESSFULLY")
print("=" * 70)

PC-GNN UNIFIED TR40 RUN
Project: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented
Configuration: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/config/pcgnn_yelpchi_unified_tr40_seed72.yml
Split: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/shared/splits/yelpchi/yelpchi_tr40_split_seed2.npz

GPU CHECK
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
GPU 0 will be used exclusively.

STARTING 100-EPOCH TR40 TRAINING
Start time: 2026-09-08T14:57:10.171487+00:00
Command: /usr/bin/python3 -u main.py --config /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/config/pcgnn_yelpchi_unified_tr40_seed72.yml
Live output will appear below.

**************** MODEL CONFIGURATION ****************
alpha                    -->   2
batch_size               -->   1024
cuda_id                  -->   0
data_dir                 -->   ./data/
data_name                -->   yelp
emb_size                 -->   64
lr                     

In [26]:
from pathlib import Path
import shutil
import subprocess
import sys
import json
import hashlib
import csv
import platform
import torch
import numpy as np

print("=" * 70)
print("PACKAGING THE COMPLETE PC-GNN TR40 RUN")
print("=" * 70)

project_dir = Path(
    "/kaggle/working/pcgnn_clean_restart/"
    "PC-GNN-COMP8851-Instrumented"
)

results_root = (
    project_dir /
    "results" /
    "unified" /
    "yelpchi" /
    "pcgnn" /
    "tr40"
)

config_path = (
    project_dir /
    "config" /
    "pcgnn_yelpchi_unified_tr40_seed72.yml"
)

split_dir = (
    project_dir /
    "shared" /
    "splits" /
    "yelpchi"
)

log_path = Path(
    "/kaggle/working/"
    "pcgnn_unified_tr40_seed72.log"
)

assert results_root.exists()
assert config_path.exists()
assert split_dir.exists()
assert log_path.exists()

summary_candidates = list(
    results_root.rglob("summary.json")
)

assert summary_candidates, (
    "No completed TR40 summary.json was found."
)

summary_path = max(
    summary_candidates,
    key=lambda path: path.stat().st_mtime,
)

run_dir = summary_path.parent

with summary_path.open("r") as file:
    summary = json.load(file)

print("Run ID:", summary["run_id"])
print("Result directory:", run_dir)
print("Configuration:", config_path)
print("Terminal log:", log_path)


print("\n" + "=" * 70)
print("STEP 1: VERIFYING REQUIRED RESULT FILES")
print("=" * 70)

required_result_files = [
    "summary.json",
    "run_config.json",
    "test_metrics.csv",
    "epoch_times.csv",
    "split_indices.npz",
    "split_summary.json",
]

for filename in required_result_files:
    path = run_dir / filename
    assert path.exists(), f"Missing required file: {path}"
    print(filename, "— FOUND")

epoch_file = run_dir / "epoch_times.csv"

with epoch_file.open("r") as file:
    epoch_rows = list(csv.DictReader(file))

assert len(epoch_rows) == 100, (
    f"Expected 100 epoch rows, found {len(epoch_rows)}"
)

print("Recorded epoch timings:", len(epoch_rows))
print("100-epoch timing check: PASS")


print("\n" + "=" * 70)
print("STEP 2: CREATING ENVIRONMENT EVIDENCE")
print("=" * 70)

evidence_dir = project_dir / "evidence"
evidence_dir.mkdir(parents=True, exist_ok=True)

environment_path = (
    evidence_dir /
    "pcgnn_unified_tr40_seed72_environment.txt"
)

environment_lines = [
    f"Python: {sys.version}",
    f"Python executable: {sys.executable}",
    f"Platform: {platform.platform()}",
    f"Processor: {platform.machine()}",
    f"PyTorch: {torch.__version__}",
    f"NumPy: {np.__version__}",
    f"CUDA available: {torch.cuda.is_available()}",
    f"CUDA version: {torch.version.cuda}",
]

if torch.cuda.is_available():
    environment_lines.extend(
        [
            f"GPU: {torch.cuda.get_device_name(0)}",
            (
                "GPU memory bytes: "
                f"{torch.cuda.get_device_properties(0).total_memory}"
            ),
        ]
    )

try:
    nvidia_output = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        check=False,
    ).stdout

    environment_lines.append("\nNVIDIA-SMI:\n")
    environment_lines.append(nvidia_output)

except Exception as error:
    environment_lines.append(
        f"\nNVIDIA-SMI unavailable: {error}"
    )

environment_path.write_text(
    "\n".join(environment_lines)
)

print("Environment evidence saved:", environment_path)


print("\n" + "=" * 70)
print("STEP 3: RECORDING SOURCE COMMIT")
print("=" * 70)

commit_path = evidence_dir / "pcgnn_source_commit.txt"

commit_result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=project_dir,
    capture_output=True,
    text=True,
    check=False,
)

if commit_result.returncode == 0:
    commit_hash = commit_result.stdout.strip()
else:
    existing_commit = project_dir / "pcgnn_commit.txt"

    if existing_commit.exists():
        commit_hash = existing_commit.read_text().strip()
    else:
        commit_hash = "Unavailable"

commit_path.write_text(commit_hash + "\n")

print("Source commit:", commit_hash)


print("\n" + "=" * 70)
print("STEP 4: CREATING THE PACKAGE")
print("=" * 70)

staging_dir = Path(
    "/kaggle/working/"
    "pcgnn_yelpchi_unified_tr40_seed72_package"
)

if staging_dir.exists():
    shutil.rmtree(staging_dir)

staging_dir.mkdir(parents=True)

shutil.copytree(
    run_dir,
    staging_dir / "results" / run_dir.name,
)

shutil.copy2(
    config_path,
    staging_dir / config_path.name,
)

shutil.copytree(
    split_dir,
    staging_dir / "shared_yelpchi_splits",
)

shutil.copy2(
    log_path,
    staging_dir / log_path.name,
)

shutil.copy2(
    environment_path,
    staging_dir / environment_path.name,
)

shutil.copy2(
    commit_path,
    staging_dir / commit_path.name,
)

metrics = summary["test_metrics"]
timing = summary["timing"]

readme_text = f"""
PC-GNN YelpChi Unified TR40 Run
================================

Run ID: {summary['run_id']}
Run mode: {summary['run_mode']}
Seed: {summary['seed']}
Best validation epoch: {summary['best_validation_epoch']}

Fixed split:
- Training: {summary['split']['train']['nodes']}
- Validation: {summary['split']['validation']['nodes']}
- Test: {summary['split']['test']['nodes']}
- Unused: {summary['split']['unused_nodes']}

Test results:
- AUROC: {metrics['auroc']:.6f}
- AUPRC: {metrics['auprc']:.6f}
- Macro-F1: {metrics['macro_f1']:.6f}
- Fraud F1: {metrics['fraud_f1']:.6f}
- Fraud precision: {metrics['fraud_precision']:.6f}
- Fraud recall: {metrics['fraud_recall']:.6f}
- G-Mean: {metrics['gmean']:.6f}

Timing:
- Epochs: {timing['training_epochs']}
- Mean epoch time: {timing['mean_train_epoch_seconds']:.6f} seconds
- Total epoch time: {timing['total_train_epoch_seconds']:.6f} seconds
- Training wall time: {timing['fit_wall_seconds_including_validation']:.6f} seconds
- Inference time: {timing['test_inference_seconds']:.6f} seconds
- Peak GPU memory: {timing['peak_gpu_memory_mb']:.6f} MB

This package contains the complete configuration, fixed split files,
100 per-epoch timing records, validation results, final test metrics,
terminal output, environment evidence and source commit.
""".strip()

(staging_dir / "README.txt").write_text(
    readme_text + "\n"
)


print("\n" + "=" * 70)
print("STEP 5: CREATING FILE CHECKSUM MANIFEST")
print("=" * 70)

manifest_rows = []

for file_path in sorted(staging_dir.rglob("*")):
    if not file_path.is_file():
        continue

    digest = hashlib.sha256(
        file_path.read_bytes()
    ).hexdigest()

    manifest_rows.append(
        {
            "file": str(
                file_path.relative_to(staging_dir)
            ),
            "size_bytes": file_path.stat().st_size,
            "sha256": digest,
        }
    )

manifest_path = staging_dir / "FILE_MANIFEST.csv"

with manifest_path.open("w", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "file",
            "size_bytes",
            "sha256",
        ],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)

print("Files recorded:", len(manifest_rows))
print("Manifest:", manifest_path)


print("\n" + "=" * 70)
print("STEP 6: CREATING DOWNLOADABLE ZIP")
print("=" * 70)

archive_base = (
    "/kaggle/working/"
    "pcgnn_yelpchi_unified_tr40_seed72_results"
)

archive_path = Path(archive_base + ".zip")

if archive_path.exists():
    archive_path.unlink()

created_archive = shutil.make_archive(
    archive_base,
    "zip",
    root_dir=staging_dir,
)

archive_sha256 = hashlib.sha256(
    Path(created_archive).read_bytes()
).hexdigest()

archive_size_mb = (
    Path(created_archive).stat().st_size /
    1024**2
)

print("ZIP file:", created_archive)
print(f"ZIP size: {archive_size_mb:.2f} MB")
print("ZIP SHA256:", archive_sha256)

print("\n" + "=" * 70)
print("TR40 EVIDENCE PACKAGE CREATED SUCCESSFULLY")
print("=" * 70)

PACKAGING THE COMPLETE PC-GNN TR40 RUN
Run ID: 20260908T145721511052Z_yelp_PCGNN_unified_yelpchi_tr40_kaggle_t4_seed72_seed72
Result directory: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/results/unified/yelpchi/pcgnn/tr40/20260908T145721511052Z_yelp_PCGNN_unified_yelpchi_tr40_kaggle_t4_seed72_seed72
Configuration: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/config/pcgnn_yelpchi_unified_tr40_seed72.yml
Terminal log: /kaggle/working/pcgnn_unified_tr40_seed72.log

STEP 1: VERIFYING REQUIRED RESULT FILES
summary.json — FOUND
run_config.json — FOUND
test_metrics.csv — FOUND
epoch_times.csv — FOUND
split_indices.npz — FOUND
split_summary.json — FOUND
Recorded epoch timings: 100
100-epoch timing check: PASS

STEP 2: CREATING ENVIRONMENT EVIDENCE
Environment evidence saved: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/evidence/pcgnn_unified_tr40_seed72_environment.txt

STEP 3: RECORDING SOURCE COMMIT
Source commit: 9d7d7fae49108117

In [4]:
from pathlib import Path
import shutil
import zipfile
import yaml
import os

print("=" * 75)
print("RESTORING PC-GNN KAGGLE WORKSPACE")
print("=" * 75)

input_root = Path("/kaggle/input")
working_root = Path("/kaggle/working")

destination = (
    working_root
    / "pcgnn_clean_restart"
    / "PC-GNN-COMP8851-Instrumented"
)

destination.parent.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Find the instrumented PC-GNN source
# ------------------------------------------------------------
print("\n[1/5] Searching Kaggle inputs for PC-GNN...")

main_candidates = list(input_root.rglob("main.py"))

pcgnn_candidates = [
    path.parent
    for path in main_candidates
    if "pc-gnn" in str(path).lower()
    or "pcgnn" in str(path).lower()
]

# Prefer the instrumented version.
pcgnn_candidates.sort(
    key=lambda path: (
        "instrumented" not in str(path).lower(),
        len(str(path))
    )
)

source_project = None

if pcgnn_candidates:
    source_project = pcgnn_candidates[0]
    print(f"Found source folder: {source_project}")

else:
    print("No extracted source folder found. Searching ZIP files...")

    zip_candidates = list(input_root.rglob("*.zip"))
    selected_zip = None

    for zip_path in zip_candidates:
        try:
            with zipfile.ZipFile(zip_path) as archive:
                names = archive.namelist()

                if any(
                    name.endswith("main.py")
                    and (
                        "pc-gnn" in name.lower()
                        or "pcgnn" in name.lower()
                    )
                    for name in names
                ):
                    selected_zip = zip_path

                    if "instrumented" in zip_path.name.lower():
                        break

        except zipfile.BadZipFile:
            continue

    assert selected_zip is not None, (
        "No PC-GNN source folder or source ZIP was found in "
        "/kaggle/input. Attach the instrumented PC-GNN dataset "
        "to this notebook and run this cell again."
    )

    print(f"Found source ZIP: {selected_zip}")

    extraction_dir = working_root / "pcgnn_restored_source"

    if extraction_dir.exists():
        shutil.rmtree(extraction_dir)

    extraction_dir.mkdir(parents=True)

    with zipfile.ZipFile(selected_zip) as archive:
        archive.extractall(extraction_dir)

    extracted_projects = [
        path.parent
        for path in extraction_dir.rglob("main.py")
        if "pc-gnn" in str(path).lower()
        or "pcgnn" in str(path).lower()
    ]

    assert extracted_projects, (
        "The ZIP was extracted, but its PC-GNN project folder "
        "could not be identified."
    )

    extracted_projects.sort(
        key=lambda path: (
            "instrumented" not in str(path).lower(),
            len(str(path))
        )
    )

    source_project = extracted_projects[0]

# ------------------------------------------------------------
# 2. Copy only the code into the writable directory
# ------------------------------------------------------------
print("\n[2/5] Copying PC-GNN code to /kaggle/working...")

if destination.exists():
    shutil.rmtree(destination)

ignored_items = shutil.ignore_patterns(
    ".git",
    ".venv",
    "__pycache__",
    "data",
    "results",
    "pytorch_models"
)

shutil.copytree(
    source_project,
    destination,
    ignore=ignored_items
)

assert (destination / "main.py").exists()

print(f"Writable project created: {destination}")

# Confirm that this is the instrumented version.
python_text = ""

for python_file in destination.rglob("*.py"):
    try:
        python_text += python_file.read_text(errors="ignore")
    except Exception:
        pass

instrumentation_found = any(
    marker in python_text
    for marker in [
        "epoch_times.csv",
        "summary.json",
        "run_config.json"
    ]
)

assert instrumentation_found, (
    "The located repository is not the instrumented PC-GNN "
    "version. Attach PC-GNN-COMP8851-Instrumented and rerun."
)

print("Instrumented source confirmed: True")

# ------------------------------------------------------------
# 3. Restore/link YelpChi data
# ------------------------------------------------------------
print("\n[3/5] Restoring YelpChi dataset files...")

data_dir = destination / "data"
data_dir.mkdir(parents=True, exist_ok=True)

required_data = [
    "YelpChi.mat",
    "yelp_homo_adjlists.pickle",
    "yelp_rur_adjlists.pickle",
    "yelp_rtr_adjlists.pickle",
    "yelp_rsr_adjlists.pickle",
]

all_zip_files = list(input_root.rglob("*.zip"))

def find_direct_file(filename):
    matches = list(input_root.rglob(filename))
    return matches[0] if matches else None

def extract_file_from_zip(filename, output_path):
    for zip_path in all_zip_files:
        try:
            with zipfile.ZipFile(zip_path) as archive:
                matching_members = [
                    member
                    for member in archive.namelist()
                    if Path(member).name == filename
                ]

                if matching_members:
                    member = matching_members[0]

                    with archive.open(member) as source:
                        with output_path.open("wb") as target:
                            shutil.copyfileobj(source, target)

                    print(
                        f"Extracted {filename} from "
                        f"{zip_path.name}"
                    )
                    return True

        except zipfile.BadZipFile:
            continue

    return False

for filename in required_data:
    target = data_dir / filename
    direct_source = find_direct_file(filename)

    if direct_source:
        if target.exists() or target.is_symlink():
            target.unlink()

        target.symlink_to(direct_source)
        print(f"Linked {filename}")
    else:
        extracted = extract_file_from_zip(filename, target)

        assert extracted, (
            f"Could not find {filename} in Kaggle inputs."
        )

# ------------------------------------------------------------
# 4. Restore the persistent YelpChi split files
# ------------------------------------------------------------
print("\n[4/5] Restoring persistent YelpChi splits...")

split_dir = (
    destination
    / "shared"
    / "splits"
    / "yelpchi"
)

split_dir.mkdir(parents=True, exist_ok=True)

split_files = [
    "yelpchi_tr40_split_seed2.npz",
    "yelpchi_tr30_split_seed2.npz",
    "yelpchi_tr20_split_seed2.npz",
    "yelpchi_tr10_split_seed2.npz",
    "yelpchi_split_summary.csv",
    "yelpchi_split_manifest.json",
]

for filename in split_files:
    target = split_dir / filename
    direct_source = find_direct_file(filename)

    if direct_source:
        shutil.copy2(direct_source, target)
        print(f"Copied {filename}")
    else:
        extracted = extract_file_from_zip(filename, target)

        assert extracted, (
            f"Could not find split file: {filename}"
        )

# ------------------------------------------------------------
# 5. Recreate the unified TR40 source configuration if needed
# ------------------------------------------------------------
print("\n[5/5] Restoring unified TR40 configuration...")

config_dir = destination / "config"
config_dir.mkdir(parents=True, exist_ok=True)

source_config = (
    config_dir
    / "pcgnn_yelpchi_unified_tr40_seed72.yml"
)

original_config = config_dir / "pcgnn_yelpchi.yml"

assert original_config.exists(), (
    "The original pcgnn_yelpchi.yml file is missing."
)

with original_config.open("r") as file:
    config = yaml.safe_load(file)

config.update({
    "data_name": "yelp",
    "data_dir": "./data/",
    "train_ratio": 0.4,
    "split_path": (
        "./shared/splits/yelpchi/"
        "yelpchi_tr40_split_seed2.npz"
    ),
    "model": "PCGNN",
    "multi_relation": "GNN",
    "emb_size": 64,
    "thres": 0.5,
    "rho": 0.5,
    "seed": 72,
    "optimizer": "adam",
    "lr": 0.01,
    "weight_decay": 0.001,
    "batch_size": 1024,
    "num_epochs": 100,
    "valid_epochs": 5,
    "alpha": 2,
    "no_cuda": False,
    "cuda_id": "0",
    "run_mode": "unified_yelpchi_tr40_kaggle_t4_seed72",
    "results_dir": "./results/unified/yelpchi/pcgnn/tr40",
})

with source_config.open("w") as file:
    yaml.safe_dump(config, file, sort_keys=False)

os.chdir(destination)

print("\n" + "=" * 75)
print("RESTORATION COMPLETE")
print("=" * 75)
print(f"Project directory: {destination}")
print(f"main.py found: {(destination / 'main.py').exists()}")
print(f"Instrumented code: {instrumentation_found}")
print(
    "YelpChi files ready:",
    all((data_dir / name).exists() for name in required_data)
)
print(
    "All split files ready:",
    all((split_dir / name).exists() for name in split_files)
)
print(f"Unified config ready: {source_config.exists()}")
print("\nWORKSPACE READY: TRUE")

RESTORING PC-GNN KAGGLE WORKSPACE

[1/5] Searching Kaggle inputs for PC-GNN...
Found source folder: /kaggle/input/datasets/maruf009/pcgnn-comp8851/PC-GNN-COMP8851-Instrumented

[2/5] Copying PC-GNN code to /kaggle/working...
Writable project created: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented
Instrumented source confirmed: True

[3/5] Restoring YelpChi dataset files...
Linked YelpChi.mat
Linked yelp_homo_adjlists.pickle
Linked yelp_rur_adjlists.pickle
Linked yelp_rtr_adjlists.pickle
Linked yelp_rsr_adjlists.pickle

[4/5] Restoring persistent YelpChi splits...
Copied yelpchi_tr40_split_seed2.npz
Copied yelpchi_tr30_split_seed2.npz
Copied yelpchi_tr20_split_seed2.npz
Copied yelpchi_tr10_split_seed2.npz
Copied yelpchi_split_summary.csv
Copied yelpchi_split_manifest.json

[5/5] Restoring unified TR40 configuration...

RESTORATION COMPLETE
Project directory: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented
main.py found: True
Instrumented code: True
Yel

In [5]:
from pathlib import Path
import subprocess
import sys
import os
import yaml
import json
import time
import torch

print("=" * 75)
print("PC-GNN TR40 HYPERPARAMETER TUNING")
print("=" * 75)

# ------------------------------------------------------------
# 1. Locate the writable PC-GNN project
# ------------------------------------------------------------
project_candidates = [
    path.parent
    for path in Path("/kaggle/working").rglob("main.py")
    if "PC-GNN" in str(path.parent) or "pcgnn" in str(path.parent).lower()
]

assert project_candidates, "No writable PC-GNN project was found."

preferred = [
    path for path in project_candidates
    if "pcgnn_clean_restart" in str(path)
]

project_dir = preferred[0] if preferred else project_candidates[0]
os.chdir(project_dir)

print(f"Project directory: {project_dir}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

assert (project_dir / "main.py").exists()
assert (project_dir / "data" / "YelpChi.mat").exists()

# ------------------------------------------------------------
# 2. Locate the completed TR40 configuration
# ------------------------------------------------------------
source_config = (
    project_dir
    / "config"
    / "pcgnn_yelpchi_unified_tr40_seed72.yml"
)

assert source_config.exists(), (
    f"Missing source configuration: {source_config}"
)

with source_config.open("r") as file:
    base_config = yaml.safe_load(file)

split_path = (
    project_dir
    / "shared"
    / "splits"
    / "yelpchi"
    / "yelpchi_tr40_split_seed2.npz"
)

assert split_path.exists(), f"Missing TR40 split: {split_path}"

print(f"Source config: {source_config}")
print(f"TR40 split: {split_path}")
print("Completed baseline: rho=0.5, seed=72")

# ------------------------------------------------------------
# 3. Define the PC-GNN tuning trials
# ------------------------------------------------------------
# The repository suggests examining these rho values.
# The already-completed rho=0.5 run is retained as the baseline.
rho_values = [0.2, 0.4, 0.6, 0.8]

config_dir = project_dir / "config" / "generated_tuning"
result_root = (
    project_dir
    / "results"
    / "unified"
    / "yelpchi"
    / "pcgnn"
    / "tuning_tr40"
)

log_dir = result_root / "logs"

config_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

trial_manifest = {
    "dataset": "YelpChi",
    "model": "PC-GNN",
    "phase": "TR40 validation tuning",
    "split_seed": 2,
    "training_seed": 72,
    "selection_metric": "validation AUROC",
    "test_used_for_selection": False,
    "fixed_parameters": {
        "optimizer": "Adam",
        "learning_rate": 0.01,
        "weight_decay": 0.001,
        "embedding_dimension": 64,
        "batch_size": 1024,
        "alpha": 2,
        "epochs": 100,
        "validation_interval": 5,
        "classification_threshold": 0.5,
    },
    "baseline_rho": 0.5,
    "additional_rho_trials": rho_values,
}

with (result_root / "tuning_manifest.json").open("w") as file:
    json.dump(trial_manifest, file, indent=2)

print("\nPlanned tuning set:")
print("  rho=0.5: already completed baseline")

for rho in rho_values:
    print(f"  rho={rho}: queued")

print("\nEstimated remaining time: approximately 90–110 minutes")
print("The full output will be written to log files.")
print("=" * 75)

# ------------------------------------------------------------
# 4. Run each trial sequentially
# ------------------------------------------------------------
environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"
environment["PYTHONUNBUFFERED"] = "1"

batch_started = time.time()
completed = []
skipped = []
failed = []

for trial_number, rho in enumerate(rho_values, start=1):
    rho_tag = str(rho).replace(".", "p")
    trial_name = f"rho_{rho_tag}"
    trial_result_dir = result_root / trial_name

    # A completed summary means this trial does not need rerunning.
    existing_summaries = list(trial_result_dir.rglob("summary.json"))

    if existing_summaries:
        print("\n" + "=" * 75)
        print(
            f"SKIPPING TRIAL {trial_number}/{len(rho_values)}: "
            f"rho={rho}"
        )
        print("Reason: a completed summary.json already exists.")
        skipped.append(trial_name)
        continue

    config = dict(base_config)

    config.update({
        "data_name": "yelp",
        "data_dir": "./data/",
        "model": "PCGNN",
        "multi_relation": "GNN",
        "split_path": (
            "./shared/splits/yelpchi/"
            "yelpchi_tr40_split_seed2.npz"
        ),
        "train_ratio": 0.4,
        "seed": 72,
        "optimizer": "adam",
        "lr": 0.01,
        "weight_decay": 0.001,
        "batch_size": 1024,
        "emb_size": 64,
        "alpha": 2,
        "rho": rho,
        "thres": 0.5,
        "num_epochs": 100,
        "valid_epochs": 5,
        "no_cuda": False,
        "cuda_id": "0",
        "run_mode": (
            f"unified_tuning_yelpchi_tr40_"
            f"rho{rho_tag}_kaggle_t4_seed72"
        ),
        "results_dir": (
            f"./results/unified/yelpchi/pcgnn/"
            f"tuning_tr40/{trial_name}"
        ),
    })

    config_path = (
        config_dir
        / f"pcgnn_yelpchi_tr40_tuning_{trial_name}_seed72.yml"
    )

    with config_path.open("w") as file:
        yaml.safe_dump(config, file, sort_keys=False)

    log_path = log_dir / f"{trial_name}_seed72.log"

    print("\n" + "=" * 75)
    print(
        f"STARTING TRIAL {trial_number}/{len(rho_values)}"
    )
    print(f"rho: {rho}")
    print(f"Config: {config_path}")
    print(f"Log: {log_path}")
    print("=" * 75)

    command = [
        sys.executable,
        "-u",
        "main.py",
        "-config",
        str(config_path.relative_to(project_dir)),
    ]

    trial_started = time.time()

    with log_path.open("w") as log_file:
        process = subprocess.Popen(
            command,
            cwd=project_dir,
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            log_file.write(line)
            log_file.flush()

            clean_line = line.strip()

            # Display useful live progress without flooding the notebook.
            if (
                clean_line.startswith("Epoch:")
                or clean_line.startswith("Valid at epoch")
                or clean_line.startswith("Restore model")
                or clean_line.startswith("F1-Macro:")
                or clean_line.startswith("AUC:")
                or clean_line.startswith("G-Mean:")
                or clean_line.startswith("Label1")
                or "Saving model" in clean_line
            ):
                print(clean_line, flush=True)

        return_code = process.wait()

    elapsed_minutes = (time.time() - trial_started) / 60

    summaries = list(trial_result_dir.rglob("summary.json"))

    if return_code == 0 and summaries:
        print("-" * 75)
        print(f"TRIAL COMPLETED: rho={rho}")
        print(f"Elapsed time: {elapsed_minutes:.2f} minutes")
        print(f"Summary: {summaries[-1]}")
        completed.append(trial_name)
    else:
        print("-" * 75)
        print(f"TRIAL FAILED: rho={rho}")
        print(f"Return code: {return_code}")
        print(f"Check log: {log_path}")
        failed.append(trial_name)
        break

# ------------------------------------------------------------
# 5. Final batch status
# ------------------------------------------------------------
batch_minutes = (time.time() - batch_started) / 60

status = {
    "completed_this_execution": completed,
    "previously_completed_and_skipped": skipped,
    "failed": failed,
    "elapsed_minutes_this_execution": batch_minutes,
}

with (result_root / "tuning_batch_status.json").open("w") as file:
    json.dump(status, file, indent=2)

print("\n" + "=" * 75)
print("TUNING BATCH STATUS")
print("=" * 75)
print(f"Completed now: {completed}")
print(f"Skipped as already complete: {skipped}")
print(f"Failed: {failed}")
print(f"Total cell time: {batch_minutes:.2f} minutes")

if not failed and len(completed) + len(skipped) == len(rho_values):
    print("\nTUNING BATCH COMPLETE: TRUE")
    print("Do not upload anything yet.")
    print(
        "The next cell will select the best rho using validation "
        "results only and prepare all final ratio/seed runs."
    )
else:
    print("\nTUNING BATCH COMPLETE: FALSE")
    print("Send me the final displayed output before proceeding.")

PC-GNN TR40 HYPERPARAMETER TUNING
Project directory: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented
Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Source config: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/config/pcgnn_yelpchi_unified_tr40_seed72.yml
TR40 split: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/shared/splits/yelpchi/yelpchi_tr40_split_seed2.npz
Completed baseline: rho=0.5, seed=72

Planned tuning set:
  rho=0.5: already completed baseline
  rho=0.2: queued
  rho=0.4: queued
  rho=0.6: queued
  rho=0.8: queued

Estimated remaining time: approximately 90–110 minutes
The full output will be written to log files.

STARTING TRIAL 1/4
rho: 0.2
Config: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/config/generated_tuning/pcgnn_yelpchi_tr40_tuning_rho_0p2_seed72.yml
Log: /kaggle/working/pcgnn_clean_restart/PC-GNN-COMP8851-Instrumented/results/unified/yelpchi/pcgnn/tuning_tr40/log

In [1]:
from pathlib import Path
import zipfile

print("=" * 75)
print("CHECKING WHETHER TUNING RESULTS SURVIVED")
print("=" * 75)

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

rho_tags = [
    "rho_0p2",
    "rho_0p4",
    "rho_0p6",
    "rho_0p8",
]

found_tags = set()
direct_results = []
zip_results = []

# ------------------------------------------------------------
# 1. Search normal folders
# ------------------------------------------------------------
print("\n[1/2] Searching normal folders...")

for root in roots:
    if not root.exists():
        continue

    for summary in root.rglob("summary.json"):
        path_text = str(summary).lower()

        for tag in rho_tags:
            if tag in path_text:
                found_tags.add(tag)
                direct_results.append(summary)
                print(f"FOUND DIRECTLY: {summary}")

# ------------------------------------------------------------
# 2. Search inside attached ZIP files
# ------------------------------------------------------------
print("\n[2/2] Searching attached ZIP files...")

zip_files = []

for root in roots:
    if root.exists():
        zip_files.extend(root.rglob("*.zip"))

for zip_path in zip_files:
    try:
        with zipfile.ZipFile(zip_path) as archive:
            for member in archive.namelist():
                member_lower = member.lower()

                if not member_lower.endswith("summary.json"):
                    continue

                for tag in rho_tags:
                    if tag in member_lower:
                        found_tags.add(tag)
                        zip_results.append((zip_path, member))
                        print(f"FOUND IN ZIP: {zip_path}")
                        print(f"  Member: {member}")

    except zipfile.BadZipFile:
        continue

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------
missing_tags = [
    tag for tag in rho_tags
    if tag not in found_tags
]

print("\n" + "=" * 75)
print("RECOVERY CHECK")
print("=" * 75)
print(f"Recovered trials: {sorted(found_tags)}")
print(f"Missing trials: {missing_tags}")

if not missing_tags:
    print("\nTUNING RESULTS RECOVERABLE: TRUE")
else:
    print("\nTUNING RESULTS RECOVERABLE: FALSE")
    print(
        "The expired Kaggle session removed some or all detailed "
        "tuning outputs."
    )

CHECKING WHETHER TUNING RESULTS SURVIVED

[1/2] Searching normal folders...

[2/2] Searching attached ZIP files...

RECOVERY CHECK
Recovered trials: []
Missing trials: ['rho_0p2', 'rho_0p4', 'rho_0p6', 'rho_0p8']

TUNING RESULTS RECOVERABLE: FALSE
The expired Kaggle session removed some or all detailed tuning outputs.
